# Computer Vision Workshop: Hand Gesture Recognition

Welcome! In this workshop, you'll learn computer vision basics and build an interactive hand gesture detection system.

## What You'll Build:
- Webcam capture and video processing
- Hand tracking with MediaPipe
- Gesture recognition (pointing, peace signs, etc.)
- Interactive meme display that reacts to your gestures!

---

## Module 1: Quick Setup

### What is Computer Vision?
Computer vision lets computers understand visual information like images and videos.

**Real-world uses:** Face unlock, self-driving cars, gesture controls, social media filters

### Key Idea: Images = Numbers
To a computer, an image is just a grid of numbers (pixels). Each pixel has a color value (0-255).

**Let's import what we need:**

In [ ]:
import cv2
import numpy as np

print("✅ Libraries imported! Ready to start.")

---
## Module 2: Working with Video and Webcam

### Understanding Video
Video = sequence of images (frames) played quickly
- 30 fps = 30 frames per second
- We process each frame like an image!

### Basic Webcam Pattern

**Note:** Webcam code runs in Python scripts (`.py` files), not notebooks. Here's the pattern you'll use:

In [ ]:
# This is the basic webcam template - copy to a .py file to run
webcam_template = '''
import cv2

# Open webcam (try 0 or 1 for camera index)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Camera not found")
    exit()

print("Camera opened! Press q to quit")

while cap.isOpened():
    success, frame = cap.read()  # Read one frame
    
    if not success:
        break
    
    # Your code here - process the frame!
    
    cv2.imshow('Webcam', frame)  # Show the frame
    
    if cv2.waitKey(1) & 0xFF == ord('q'):  # Press q to quit
        break

cap.release()  # Close camera
cv2.destroyAllWindows()  # Close windows
'''

print(webcam_template)

### Drawing on Frames

You can draw shapes and text on each frame. Here are the essentials:

In [ ]:
drawing_guide = '''
# Draw a circle (perfect for marking points)
cv2.circle(frame, (x, y), radius, (0, 255, 0), thickness)
#                center   radius  color(BGR)  -1=filled, 2=outline

# Draw text
cv2.putText(frame, "POINTING!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 
            1, (0, 0, 255), 2)
#           text          position  font              size  color   thickness

# Draw rectangle
cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)

# Draw line
cv2.line(frame, (x1, y1), (x2, y2), (255, 255, 0), 2)
'''

print(drawing_guide)
print("\n💡 Important: Color format is BGR (Blue, Green, Red), not RGB!")
print("   (0, 255, 0)   = Green")
print("   (255, 0, 0)   = Blue")
print("   (0, 0, 255)   = Red")
print("   (255, 255, 0) = Cyan")

### ✏️ Exercise 1: Create Your First Webcam Script

Create a file called `webcam_test.py` with this code:

```python
import cv2

cap = cv2.VideoCapture(0)  # or try 1

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # TODO: Draw a green circle in the center
    h, w, _ = frame.shape
    cv2.circle(frame, (w//2, h//2), 50, (0, 255, 0), 3)
    
    # TODO: Add text showing frame size
    cv2.putText(frame, f"{w}x{h}", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow('Webcam Test', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
```

Run it: `python webcam_test.py`

---
## Module 3: MediaPipe Hand Tracking

### What is MediaPipe?

MediaPipe is Google's pre-trained machine learning framework. Instead of training our own models (takes weeks!), we use their models:
- **Hand tracking** - 21 landmarks per hand
- Face detection
- Pose estimation
- And more!

### The 21 Hand Landmarks

```
Landmark indices:

       8   12  16  20    (fingertips)
       |   |   |   |
   4   7  11  15  19
   |   |   |   |   |
   3   6  10  14  18
   |   |   |   |   |
   2   5   9  13  17    (knuckles)
    \  |   |   |   /
     \ |   |   |  /
        0 (wrist)
```

**Key landmarks you'll use:**
- `hand[0]` = Wrist
- `hand[4]` = Thumb tip
- `hand[8]` = Index finger tip
- `hand[12]` = Middle finger tip
- `hand[16]` = Ring finger tip
- `hand[20]` = Pinky tip

**For finger joints:**
- Tip: 8, 12, 16, 20
- PIP (middle joint): 6, 10, 14, 18  ← **Use this to detect if finger is up/down!**
- MCP (knuckle): 5, 9, 13, 17

### Setting Up MediaPipe

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Configure hand tracking
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,                              # Detect up to 2 hands
    running_mode=vision.RunningMode.VIDEO,    # For video/webcam
    min_hand_detection_confidence=0.6,        # Confidence threshold (0-1)
    min_tracking_confidence=0.6
)

landmarker = vision.HandLandmarker.create_from_options(options)

print("✅ MediaPipe Hand Landmarker ready!")
print(f"  - Detecting up to {options.num_hands} hands")
print(f"  - Confidence threshold: {options.min_hand_detection_confidence}")

### How to Detect Hands in Your Webcam Loop

In [ ]:
detection_code = '''
timestamp = 0  # Initialize before loop

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # Convert BGR to RGB (MediaPipe needs RGB)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    # Detect hands
    result = landmarker.detect_for_video(mp_image, timestamp)
    timestamp += 1
    
    # Check if hands detected
    if result.hand_landmarks:
        for hand in result.hand_landmarks:  # Loop through each detected hand
            # hand is a list of 21 landmarks
            
            # Draw all landmarks
            for landmark in hand:
                # Landmarks are normalized (0 to 1)
                # Convert to pixel coordinates
                h, w, _ = frame.shape
                cx = int(landmark.x * w)
                cy = int(landmark.y * h)
                
                # Draw green circle on each landmark
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
    
    cv2.imshow('Hand Detection', frame)
'''

print(detection_code)

### Understanding Landmark Coordinates

Each landmark has:
- `landmark.x` - Horizontal position (0=left, 1=right)
- `landmark.y` - Vertical position (0=top, 1=bottom)
- `landmark.z` - Depth (distance from camera)

**These are normalized (0 to 1)**, so they work with any image size!

In [ ]:
# Accessing specific landmarks
landmark_access = '''
if result.hand_landmarks:
    for hand in result.hand_landmarks:
        # Get specific landmarks by index
        wrist = hand[0]
        thumb_tip = hand[4]
        index_tip = hand[8]
        index_pip = hand[6]  # Middle joint
        middle_tip = hand[12]
        
        # Print coordinates
        print(f"Index tip: x={index_tip.x:.2f}, y={index_tip.y:.2f}")
        print(f"Index pip: x={index_pip.x:.2f}, y={index_pip.y:.2f}")
'''

print(landmark_access)

### ✏️ Exercise 2: Hand Detection Script

Create `hand_detection.py` that:
1. Opens webcam
2. Detects hands with MediaPipe
3. Draws green circles on all 21 landmarks
4. Shows "Hand Detected!" text when a hand is found

Combine the webcam template + MediaPipe detection code!

---
## Module 4: Hand Gesture Recognition

### The Logic: How to Detect Gestures

To detect gestures, **compare landmark positions**.

**Key Insight for Fingers:**
- If fingertip.y **< ** joint.y → Finger is **UP** (extended)
- If fingertip.y **>** joint.y → Finger is **DOWN** (folded)

**Why y-coordinates?**
- y=0 is at the **top** of the screen
- y=1 is at the **bottom** of the screen
- So smaller y = higher position

### Example 1: Pointing Finger

**Logic:**
- Index finger UP (extended)
- Middle, ring, pinky DOWN (folded)

In [ ]:
pointing_code = '''
if result.hand_landmarks:
    for hand in result.hand_landmarks:
        # Get fingertips and joints
        index_tip = hand[8]
        index_pip = hand[6]    # Middle joint of index
        
        middle_tip = hand[12]
        middle_pip = hand[10]
        
        ring_tip = hand[16]
        ring_pip = hand[14]
        
        pinky_tip = hand[20]
        pinky_pip = hand[18]
        
        # Check finger states
        index_extended = index_tip.y < index_pip.y   # Tip above joint = UP
        middle_folded = middle_tip.y > middle_pip.y  # Tip below joint = DOWN
        ring_folded = ring_tip.y > ring_pip.y
        pinky_folded = pinky_tip.y > pinky_pip.y
        
        # Detect pointing gesture
        if index_extended and middle_folded and ring_folded and pinky_folded:
            print("👉 POINTING!")
            cv2.putText(frame, "POINTING!", (50, 50),
                       cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)
'''

print(pointing_code)

### Example 2: Peace Sign ✌️

**Logic:**
- Index and middle fingers UP
- Ring and pinky DOWN

In [ ]:
peace_code = '''
index_extended = index_tip.y < index_pip.y
middle_extended = middle_tip.y < middle_pip.y
ring_folded = ring_tip.y > ring_pip.y
pinky_folded = pinky_tip.y > pinky_pip.y

if index_extended and middle_extended and ring_folded and pinky_folded:
    print("✌️ PEACE!")
    cv2.putText(frame, "PEACE!", (50, 50),
               cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 0, 255), 3)
'''

print(peace_code)

### Example 3: Counting Fingers

Count how many fingers are extended:

In [ ]:
counting_code = '''
fingers_up = 0

# Check each finger (excluding thumb for simplicity)
if index_tip.y < index_pip.y:
    fingers_up += 1
if middle_tip.y < middle_pip.y:
    fingers_up += 1
if ring_tip.y < ring_pip.y:
    fingers_up += 1
if pinky_tip.y < pinky_pip.y:
    fingers_up += 1

print(f"Fingers: {fingers_up}")
cv2.putText(frame, f"Fingers: {fingers_up}", (50, 50),
           cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3)
'''

print(counting_code)

### Example 4: Thumbs Up 👍

**Note:** Thumb is tricky! It moves horizontally, not vertically.

In [ ]:
thumbs_code = '''
thumb_tip = hand[4]
thumb_mcp = hand[2]  # Thumb base

# Thumb extended (uses x-direction for horizontal movement)
thumb_extended = abs(thumb_tip.x - thumb_mcp.x) > 0.1

# All other fingers folded
index_folded = index_tip.y > index_pip.y
middle_folded = middle_tip.y > middle_pip.y
ring_folded = ring_tip.y > ring_pip.y
pinky_folded = pinky_tip.y > pinky_pip.y

if thumb_extended and index_folded and middle_folded and ring_folded and pinky_folded:
    print("👍 THUMBS UP!")
'''

print(thumbs_code)

### ✏️ Exercise 3: Multi-Gesture Detection

Modify your `hand_detection.py` to detect:
1. Pointing finger
2. Peace sign
3. Show different text/colors for each gesture

**Bonus challenges:**
- Add finger counting
- Detect thumbs up
- Detect a fist (all fingers down)

---
## Module 5: Interactive Meme Display

### The Final Project!

Now let's build the complete gesture-controlled meme app:
1. Webcam feed
2. Hand detection
3. Gesture recognition
4. Different memes for different gestures
5. Side-by-side display (meme + webcam)

### Step 1: Load Multiple Meme Images

In [ ]:
loading_memes = '''
# Load all memes at startup (more efficient than loading in loop)
meme_staring = cv2.imread("meme/staring.png")
meme_pointing = cv2.imread("meme/pointing.png")
# Add more memes here

# Verify they loaded
if meme_staring is None:
    print("Error: Could not load staring.png")
    exit()
if meme_pointing is None:
    print("Error: Could not load pointing.png")
    exit()

# Start with default meme
current_meme = meme_staring
'''

print(loading_memes)

### Step 2: Switch Memes Based on Gestures

In [ ]:
switching_memes = '''
# Inside the webcam loop, after detecting hands:
if result.hand_landmarks:
    for hand in result.hand_landmarks:
        # Get landmarks...
        
        # Check for pointing
        if index_extended and middle_folded and ring_folded and pinky_folded:
            current_meme = meme_pointing
            cv2.putText(frame, "POINTING!", (50, 50), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        else:
            current_meme = meme_staring
        
        # Draw landmarks...
else:
    # No hand detected
    current_meme = meme_staring
'''

print(switching_memes)

### Step 3: Combine Images Side-by-Side

In [ ]:
combining_images = '''
# After processing the frame, before displaying:

# Resize meme to match webcam frame size
frame_height, frame_width = frame.shape[:2]
meme_resized = cv2.resize(current_meme, (frame_width, frame_height))

# Stack horizontally (side by side)
combined = np.hstack([meme_resized, frame])
#                      [left image,   right image]

# Display combined image
cv2.imshow('Think Monke', combined)
'''

print(combining_images)
print("\n💡 np.hstack() = horizontal stack (side by side)")
print("   np.vstack() = vertical stack (top to bottom)")

### Complete Application Code

Here's everything put together:

In [ ]:
complete_app = '''
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ===== LOAD MEME IMAGES =====
meme_staring = cv2.imread("meme/staring.png")
meme_pointing = cv2.imread("meme/pointing.png")

if meme_staring is None or meme_pointing is None:
    print("Error: Could not load meme images")
    exit()

current_meme = meme_staring

# ===== SETUP MEDIAPIPE =====
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    running_mode=vision.RunningMode.VIDEO,
    min_hand_detection_confidence=0.6,
    min_tracking_confidence=0.6
)
landmarker = vision.HandLandmarker.create_from_options(options)

# ===== OPEN WEBCAM =====
cap = cv2.VideoCapture(1)  # Try 0 if 1 doesn't work
timestamp = 0

if not cap.isOpened():
    print("Error: Could not open camera")
    exit()

print("Camera opened! Press 'q' to quit")

# ===== MAIN LOOP =====
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # Detect hands
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    result = landmarker.detect_for_video(mp_image, timestamp)
    timestamp += 1
    
    # Process gestures
    if result.hand_landmarks:
        for hand in result.hand_landmarks:
            # Get landmarks
            index_tip = hand[8]
            index_pip = hand[6]
            middle_tip = hand[12]
            middle_pip = hand[10]
            ring_tip = hand[16]
            ring_pip = hand[14]
            pinky_tip = hand[20]
            pinky_pip = hand[18]
            
            # Check gestures
            index_extended = index_tip.y < index_pip.y
            middle_folded = middle_tip.y > middle_pip.y
            ring_folded = ring_tip.y > ring_pip.y
            pinky_folded = pinky_tip.y > pinky_pip.y
            
            if index_extended and middle_folded and ring_folded and pinky_folded:
                current_meme = meme_pointing
                cv2.putText(frame, "POINTING!", (50, 50),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            else:
                current_meme = meme_staring
            
            # Draw landmarks
            for lm in hand:
                h, w, _ = frame.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
    else:
        current_meme = meme_staring
    
    # Combine meme and webcam
    frame_height, frame_width = frame.shape[:2]
    meme_resized = cv2.resize(current_meme, (frame_width, frame_height))
    combined = np.hstack([meme_resized, frame])
    
    # Display
    cv2.imshow('Think Monke', combined)
    
    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

# ===== CLEANUP =====
cap.release()
cv2.destroyAllWindows()
landmarker.close()
'''

print("=" * 60)
print("COMPLETE APPLICATION CODE")
print("=" * 60)
print(complete_app)
print("\nSave this as 'gesture_meme.py' and run it!")

### ✏️ Final Exercise: Build Your Own!

Extend the app with your own ideas:

**Easy additions:**
- Add more memes for peace sign, fist, etc.
- Change the text color per gesture
- Add a gesture counter

**Medium challenges:**
- Detect both hands and compare their gestures
- Add sound effects for each gesture
- Create a gesture-based game (rock-paper-scissors)

**Advanced projects:**
- Virtual drawing: track index finger to draw lines
- Gesture sequence detector (do 3 gestures in order)
- Hand gesture music player controls
- Virtual buttons activated by finger touches

---
## Debugging Tips

### Camera Issues
```python
# Try different camera indices
cap = cv2.VideoCapture(0)  # Built-in
cap = cv2.VideoCapture(1)  # External
```

### Image Loading Issues
```python
img = cv2.imread("meme/staring.png")
if img is None:
    print("Error: Check file path!")
```

### Hand Detection Not Working
- **Lighting:** Need good lighting!
- **Lower threshold:** `min_hand_detection_confidence=0.3`
- **Hand visible:** Make sure full hand is in frame
- **Check result:** `if result.hand_landmarks:`

### Gesture Detection Flaky
```python
# Add tolerance
index_extended = index_tip.y < index_pip.y - 0.02

# Print debug info
print(f"Index tip: {index_tip.y:.3f}")
print(f"Index pip: {index_pip.y:.3f}")
print(f"Extended: {index_tip.y < index_pip.y}")
```

### Common Mistakes
- Forgetting to convert BGR → RGB for MediaPipe
- Not incrementing timestamp
- Using wrong landmark indices
- Comparing x instead of y for vertical movements

---
## Summary & Next Steps

### 🎉 What You Learned:

✅ **Computer vision basics** - Images as numbers

✅ **Video processing** - Webcam capture, frame-by-frame processing

✅ **MediaPipe hand tracking** - 21 landmarks, pre-trained models

✅ **Gesture recognition** - Comparing landmark positions

✅ **Interactive apps** - Combining images, real-time responses

### 🚀 Next Steps:

1. **More MediaPipe models:**
   - Face detection
   - Pose estimation (full body tracking)
   - Object detection

2. **Advanced gestures:**
   - Pinch detection (thumb + index distance)
   - Hand orientation (palm up/down)
   - Dynamic gestures (swipes, waves)

3. **Build projects:**
   - Virtual piano
   - Gesture-based game controls
   - Sign language detector
   - Virtual whiteboard

4. **Learn more:**
   - Machine learning basics
   - Training custom models
   - Performance optimization

### 📚 Resources:

- [OpenCV Docs](https://docs.opencv.org/)
- [MediaPipe Solutions](https://developers.google.com/mediapipe/solutions/guide)
- [NumPy Tutorial](https://numpy.org/doc/stable/user/quickstart.html)

---

## You did it! 🎊

You can now build gesture-controlled computer vision apps. Keep experimenting and building cool stuff! 🚀